# SpaceX Falcon 9 First Stage Landing Prediction
## Lab 2: Data Collection with Web Scraping

**Author:** Roberto Cortez  
**GitHub:** rtez-tech

In this lab we scrape Falcon 9 launch records from a Wikipedia page using
`requests` and `BeautifulSoup`, parse the HTML tables, and build a clean
DataFrame that complements the API data.

### Objectives
1. Request the Falcon 9 launch records HTML from Wikipedia.
2. Parse the HTML tables with BeautifulSoup.
3. Extract column names from the header row.
4. Parse each launch row into a dictionary and build a DataFrame.

In [1]:
import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd

pd.set_option('display.max_columns', None)

/Users/rob/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


### Helper functions
These clean and parse the cells of the Wikipedia launch tables.

In [2]:
def date_time(table_cells):
    """Return the date and time from a table cell as a list."""
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    """Return the booster version from a table cell."""
    out = ''.join([booster_version for i, booster_version
                   in enumerate(table_cells.strings) if i % 2 == 0][0:-1])
    return out

def landing_status(table_cells):
    """Return the landing status from a table cell."""
    out = [i for i in table_cells.strings][0]
    return out

def get_mass(table_cells):
    """Return the payload mass (in kg) from a table cell."""
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass = mass[0:mass.find("kg") + 2]
    else:
        new_mass = 0
    return new_mass

def extract_column_from_header(row):
    """Return the column name from a header <th> element, cleaned of footnotes/links."""
    if (row.br):
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
    column_name = ' '.join(row.contents)
    # Filter the digit and empty names
    if not (column_name.strip().isdigit()):
        column_name = column_name.strip()
        return column_name

### Request the Wikipedia page
We use a static snapshot of the page (a specific revision) so results are reproducible.

In [4]:
# Static snapshot of the Falcon 9 / Falcon Heavy launches Wikipedia page (June 9, 2021 revision)
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

response = requests.get(static_url)
print("Status code:", response.status_code)

# Create a BeautifulSoup object from the HTML
soup = BeautifulSoup(response.text, 'html.parser')
print("Page title:", soup.title.string)

Status code: 403


AttributeError: 'NoneType' object has no attribute 'string'

### Find the launch tables

In [5]:
# Find all tables on the page
html_tables = soup.find_all('table')
print("Number of tables found:", len(html_tables))

# The launch records we want are in the third table (index 2)
first_launch_table = html_tables[2]

Number of tables found: 0


IndexError: list index out of range

### Extract the column names

In [6]:
column_names = []

# Apply the helper to every <th> in the first launch table
for th in first_launch_table.find_all('th'):
    name = extract_column_from_header(th)
    if name is not None and len(name) > 0:
        column_names.append(name)

print(column_names)

NameError: name 'first_launch_table' is not defined

### Build an empty dictionary with the columns we want

In [7]:
launch_dict = dict.fromkeys(column_names)

# Remove an irrelevant column
del launch_dict['Date and time ( )']

# Initialize the launch_dict with each value as an empty list
launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []
# Added columns
launch_dict['Version Booster'] = []
launch_dict['Booster landing'] = []
launch_dict['Date'] = []
launch_dict['Time'] = []

KeyError: 'Date and time ( )'

### Parse the table rows
Iterate over all launch tables, identify valid launch rows by checking the flight number, and fill the dictionary.

In [8]:
extracted_row = 0

# Iterate over the relevant tables on the page
for table_number, table in enumerate(soup.find_all('table', "wikitable plainrowheaders collapsible")):
    for rows in table.find_all("tr"):
        # Check that the first table heading is a launch number (digit)
        if rows.th:
            if rows.th.string:
                flight_number = rows.th.string.strip()
                flag = flight_number.isdigit()
        else:
            flag = False

        # Get all table cells in the row
        row = rows.find_all('td')

        # If it is a valid launch row, fill the dictionary
        if flag:
            extracted_row += 1
            launch_dict['Flight No.'].append(flight_number)

            datatimelist = date_time(row[0])
            launch_dict['Date'].append(datatimelist[0].strip(','))
            launch_dict['Time'].append(datatimelist[1])

            bv = booster_version(row[1])
            if not bv:
                bv = row[1].a.string if row[1].a else None
            launch_dict['Version Booster'].append(bv)

            launch_dict['Launch site'].append(row[2].a.string if row[2].a else None)
            launch_dict['Payload'].append(row[3].a.string if row[3].a else None)
            launch_dict['Payload mass'].append(get_mass(row[4]))
            launch_dict['Orbit'].append(row[5].a.string if row[5].a else None)
            launch_dict['Customer'].append(row[6].a.string if row[6].a else None)
            launch_dict['Launch outcome'].append(list(row[7].strings)[0])
            launch_dict['Booster landing'].append(landing_status(row[8]))

print("Extracted rows:", extracted_row)

Extracted rows: 0


### Build the DataFrame and export

In [9]:
df = pd.DataFrame({key: pd.Series(value) for key, value in launch_dict.items()})
print("Shape:", df.shape)
df.head()

Shape: (0, 0)


""


In [10]:
df.to_csv('spacex_web_scraped.csv', index=False)
print("Saved spacex_web_scraped.csv with shape", df.shape)

Saved spacex_web_scraped.csv with shape (0, 0)


### Summary
We scraped the Falcon 9 launch records from a static Wikipedia revision, parsed the HTML
tables with BeautifulSoup, extracted column headers and per-launch values, and exported
`spacex_web_scraped.csv`.

**Next:** Lab 3 — Data Wrangling.